## Assumptions:
- Invalid timestamps are rejected.
- Missing active_power or setpoint values are rejected.
- Duplicate records are removed.
- Numeric values may contain thousands separators.
- Bad rows are logged rather than causing the pipeline to fail.

In [50]:
import pandas as pd
from pathlib import Path
import logging

#Config
RAW_PATH = Path("../task_2_assets/data/telemetry_raw.csv") 
CLEAN_PATH = Path("../task_2_assets/data/telemetry_cleaned.csv") 
REJECT_PATH = Path("../task_2_assets/data/rejected_rows.csv") 
logging.basicConfig( level=logging.INFO, format="%(levelname)s - %(message)s" ) 
logger = logging.getLogger(__name__)

In [51]:
raw = pd.read_csv(RAW_PATH, thousands=",") 
logger.info(f"Loaded {len(raw)} rows from {RAW_PATH}") 
df = raw.copy() 

INFO - Loaded 12 rows from ..\task_2_assets\data\telemetry_raw.csv


In [ ]:
# Type conversions 
df["timestamp"] = pd.to_datetime( df["timestamp"], format="%d/%m/%Y %H:%M:%S", errors="coerce", ) 
for column in ["active_power", "setpoint", "site_id"]: 
    df[column] = pd.to_numeric(df[column], errors="coerce") 

In [64]:
# Validation
def validate(frame: pd.DataFrame) -> dict:
    rules = {
        "Valid timestamp": frame["timestamp"].notna(),
        "Active power present": frame["active_power"].notna(),
        "Setpoint present": frame["setpoint"].notna(),
        "Site ID present": frame["site_id"].notna(),
        "Site ID positive": frame["site_id"].fillna(-1) > 0,
        "Active power non-negative": frame["active_power"].fillna(-1) >= 0,
        "Setpoint non-negative": frame["setpoint"].fillna(-1) >= 0,
        "Not duplicate": ~frame.duplicated(
            subset=["timestamp", "site_id"],
            keep="first"
        ),
    }
    return rules


def report_validation_results(rules):
    all_passed = True
    for name, passed in rules.items():
        failed = (~passed).sum()
        if failed:
            logger.warning(f"{name}: {failed} row(s) failed")
            all_passed = False
        else:
            logger.info(f"{name}: PASS")
    return all_passed 

In [57]:
# Validate raw data
logger.info("Validation before cleaning") 
rules = validate(df) 
report_validation_results(rules) 
validation = pd.DataFrame(rules)

INFO - Validation before cleaning
WARNING - Valid timestamp: 1 row(s) failed
WARNING - Active power present: 1 row(s) failed
WARNING - Setpoint present: 1 row(s) failed
INFO - Site ID present: PASS
INFO - Site ID positive: PASS
WARNING - Active power non-negative: 1 row(s) failed
WARNING - Setpoint non-negative: 1 row(s) failed
WARNING - Not duplicate: 1 row(s) failed


In [59]:
# Create rejection report
invalid_rows = df.loc[~validation.all(axis=1)].copy()

if not invalid_rows.empty:
    invalid_validation = validation.loc[invalid_rows.index]

    invalid_rows["rejection_reason"] = invalid_validation.apply(
        lambda row: ", ".join(row.index[~row]),
        axis=1,
    )

    invalid_rows.to_csv(REJECT_PATH, index=False)

    logger.warning(
        f"Rejected {len(invalid_rows)} row(s). "
        f"Saved to {REJECT_PATH}"
    )

WARNING - Rejected 4 row(s). Saved to ..\task_2_assets\data\rejected_rows.csv


In [60]:
# Keep only valid rows
clean = (
    df.loc[validation.all(axis=1)]
      .sort_values("timestamp")
      .reset_index(drop=True)
)

logger.info(f"Input rows : {len(df)}")
logger.info(f"Valid rows : {len(clean)}")
logger.info(f"Rejected   : {len(df) - len(clean)}")

INFO - Input rows : 12
INFO - Valid rows : 8
INFO - Rejected   : 4


In [61]:
# Final validation

logger.info("Validation after cleaning")
clean_rules = validate(clean)
clean_is_valid = report_validation_results(clean_rules)
assert clean_is_valid, (
    "Cleaned dataset failed validation."
)





INFO - Validation after cleaning
INFO - Valid timestamp: PASS
INFO - Active power present: PASS
INFO - Setpoint present: PASS
INFO - Site ID present: PASS
INFO - Site ID positive: PASS
INFO - Active power non-negative: PASS
INFO - Setpoint non-negative: PASS
INFO - Not duplicate: PASS


In [62]:
# Save cleaned data
clean.to_csv(CLEAN_PATH, index=False)
logger.info(f"Saved cleaned data to {CLEAN_PATH}")

INFO - Saved cleaned data to ..\task_2_assets\data\telemetry_cleaned.csv


In [63]:

# Quick verification
print("\nDataset info")
print(clean.info())
print("\nSummary statistics")
print(clean.describe(include="all"))
print("\nCleaned data")
display(clean)


Dataset info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   timestamp     8 non-null      datetime64[ns]
 1   active_power  8 non-null      float64       
 2   setpoint      8 non-null      float64       
 3   site_id       8 non-null      int64         
dtypes: datetime64[ns](1), float64(2), int64(1)
memory usage: 384.0 bytes
None

Summary statistics
                 timestamp  active_power      setpoint  site_id
count                    8      8.000000      8.000000      8.0
mean   2026-07-01 02:15:00  18456.250000  18781.250000      1.0
min    2026-07-01 00:00:00  12500.000000  18000.000000      1.0
25%    2026-07-01 00:52:30  18387.500000  18250.000000      1.0
50%    2026-07-01 02:15:00  19225.000000  18875.000000      1.0
75%    2026-07-01 03:37:30  19737.500000  19262.500000      1.0
max    2026-07-01 04:30:00  20500.00000

,timestamp,active_power,setpoint,site_id
0,2026-07-01 00:00:00,18200.0,18000.0,1
1,2026-07-01 00:30:00,18450.0,18100.0,1
2,2026-07-01 01:00:00,12500.0,18300.0,1
3,2026-07-01 02:00:00,19100.0,18800.0,1
4,2026-07-01 02:30:00,19350.0,18950.0,1
5,2026-07-01 03:30:00,19700.0,19250.0,1
6,2026-07-01 04:00:00,19850.0,19300.0,1
7,2026-07-01 04:30:00,20500.0,19550.0,1


## Summary

The raw CSV was loaded using pandas

The data was cleaned by:
- Ghousands separators handled during import using `thousands=","`.
- Parsing timestamps using the expected `DD/MM/YYYY HH:MM:SS` format.
- Converting numeric fields.
- Coercing invalid values to `NaN` rather than causing the pipeline to fail.
- Validating required fields (`timestamp`, `active_power`, `setpoint`, and `site_id`).
- Validating that `site_id` values are positive and that power values are non-negative.
- Identifying and removing duplicate records based on the business key (`timestamp`, `site_id`).
- Logging validation results.
- Exporting rejected rows to `rejected_rows.csv`, including the reason(s) each row was rejected.

Only rows that passed all validation rules were retained. The cleaned dataset was sorted by timestamp, re-validated to ensure it was safe to load, and then saved to `telemetry_cleaned.csv`.
This approach ensures that invalid or malformed records are handled gracefully without interrupting the pipeline, while maintaining an audit trail of rejected data.